# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Note: metadata is an object, use attributes.
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n\nIdentifier: {metadata.identifier}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and fields, referencing them by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    {f.name} (@id: {f.id}, dataType: {f.data_type})")
    print('---')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
import collections

# List record set IDs (@id)
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    # records should be List[Dict], safe to pd.DataFrame
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet: {record_set_id}")
    else:
        print(f"No records found in RecordSet: {record_set_id}")

# For demonstration, select the 'main' record set for further analysis.
main_record_set_id = record_set_ids[0]

print(f"Available columns in main record set ({main_record_set_id}): ")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
main_df = dataframes[main_record_set_id]

# Examine numeric fields in the main record set (@id and types in section 2)
numeric_field_choices = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
print(f"Numeric fields detected in `{main_record_set_id}`:", numeric_field_choices)
# Example: choose 'Coverage' if available, else first numeric field.
example_numeric_field = None
for candidate in ['Coverage', 'PercentCoverage', 'percent_coverage', 'peptide_count', 'peptide_counts', 'mw', 'MW']:
    if candidate in main_df.columns:
        example_numeric_field = candidate
        break
if not example_numeric_field and numeric_field_choices:
    example_numeric_field = numeric_field_choices[0]

if example_numeric_field:
    threshold = main_df[example_numeric_field].mean()
    filtered_df = main_df[main_df[example_numeric_field] > threshold].copy()
    print(f"Filtered records with {example_numeric_field} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{example_numeric_field}_normalized"] = (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) / filtered_df[example_numeric_field].std()
    print(f"\nNormalized {example_numeric_field} for filtered records:")
    print(filtered_df[[example_numeric_field, f"{example_numeric_field}_normalized"]].head())

    # Group by a categorical field, if available
    group_field = None
    for candidate in ['Group', 'group', 'protein_group', 'Sample', 'sample']:
        if candidate in main_df.columns:
            group_field = candidate
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[example_numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean {example_numeric_field}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Make a simple histogram and boxplot of the chosen numeric field
if example_numeric_field:
    plt.figure(figsize=(10,4))
    sns.histplot(main_df[example_numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {example_numeric_field}')
    plt.xlabel(example_numeric_field)
    plt.show()

    plt.figure(figsize=(6,4))
    sns.boxplot(x=main_df[example_numeric_field].dropna())
    plt.title(f'Boxplot of {example_numeric_field}')
    plt.show()

    # If group_field exists, plot group comparison
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=main_df[group_field], y=main_df[example_numeric_field])
        plt.title(f'{example_numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains mass spectrometry data with protein-level details (e.g., coverage, peptide counts, molecular weight).  
- Used the `mlcroissant` library to load metadata, enumerate schemas (record sets and fields), and explored record set data using DataFrames.  
- Performed basic exploratory analysis on numeric fields (such as filtering by mean and normalization), and visualized distributions using common plots.  
- Data structure and `@id` field referencing facilitate robust and reproducible data access using Croissant schemas.  

_You can now use the filtered and grouped data for further downstream analysis relevant to protein abundance, identification, or comparative studies!_